In [ ]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

import time
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP,SELECT
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar

In [ ]:
chip=CHIP(PS(host="192.168.1.11", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0,IsNew32=True)
chip.adc.set_gap(adc_cs_gap=90,adc_first_gap=10,adc_last_gap=10)
chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
chip.clk_manager.set_cyc(20, 20,delay3=100)
chip.add_compiler("../compiler/code/")

In [ ]:
# set_good_device_file_name = "../data/good_device/cond_500_r8.npy"
# reset_good_device_file_name = "../data/good_device/cond_200_r8.npy"

In [ ]:
select = SELECT()

### 1. Reset操作

In [ ]:
select.Reset(chip=chip,need_read = np.ones((256,256),dtype=bool),write_times=41,start_v=1,delta_v=0.05,tg=5,threshold=150,
      reset_pulse_width=100e-6,read_type=2,sub_base=True,plot_cond=plot_cond,vmax=1400)

In [ ]:
crossbar = np.ones((256,256))
_,cond,_ = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
plot_cond(cond)

In [ ]:
print(np.sum(cond<200))
reset_good_device_file_name = "../data/good_device/20250613_cond_200_r8.npy"
np.save(reset_good_device_file_name,cond<200)

### 2. Set操作

In [ ]:
need_read = np.ones((256,256),dtype=bool)
# need_read[:20,120:180]=True
select.Set(chip=chip,need_read=need_read,write_times=1,write_voltage=5,start_tg=1.5,delta_tg=0.05,threshold=800,set_pulse_width=100e-6,sub_base=True,vmax=1000,plot_cond=plot_cond)

In [ ]:
crossbar = np.ones((256,256))
_,cond,_ = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
plot_cond(cond)

In [ ]:
print(np.sum(cond>500))
set_good_device_file_name = "../data/good_device/20250613_cond_500_r8.npy"
np.save(set_good_device_file_name,cond>500)